# 01 — Clean: load, map columns, assert, write

**Consumes** `data/TCAS{66,67,68,69}_maxmin.xlsx` — four รอบ 3 Admission
min/max score files, 13/16/16/18 columns, no two years spelled alike.

**Emits**
- `data/processed/admission_long.parquet` (+ `.csv`) — 19,437 tidy rows, one
  stable schema across all four years.
- `data/processed/chula_panel.csv` — the balanced 48-code × 4-year panel.

**Concludes** the four files can be put on one schema without fuzzy matching:
Chula's 14 faculty names are byte-identical across all four years and all 48
of its TCAS66 program codes survive to TCAS69. The headline series is
*first-processing* only; Double Sorting is carried separately as `*_ds` so it
can never leak into the main series. Three invariants are asserted rather than
hoped for, and two known source defects are logged rather than patched.

Nothing downstream is trustworthy until this notebook's asserts are green.

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd

from src.load import (
    YEAR_COLUMNS, SCHEMA, CHULA,
    load_all, chula, chula_panel, validate,
    reconstructed_seats, stable_chula_codes, write, PROCESSED,
)

pd.set_option("display.width", 200)

## 1. The column mapping

Every year-specific column name lives in one dict, `src.load.YEAR_COLUMNS`.
The only series comparable across all four years is **first-processing**
(ประมวลผลครั้งที่ 1). TCAS66 has no second-pass columns at all — ทปอ. ran
Double Sorting that year but never published it.

In [2]:
mapping = pd.DataFrame(YEAR_COLUMNS).drop(index="sheet")
mapping.loc[["faculty", "program", "passed", "min_score", "min_score_ds"]]

,66,67,68,69
faculty,คณะ/สำนักวิชา,คณะ,คณะ,คณะ
program,ชื่อหลักสูตร,หลักสูตร,หลักสูตร,หลักสูตร
passed,ผ่าน,ผ่าน(รอบ1),ผ่าน ประมวลผลครั้งที่ 1,ผ่าน
min_score,คะแนนต่ำสุด,คะแนนต่ำสุด,คะแนนต่ำสุด ประมวลผลครั้งที่ 1,คะแนนต่ำสุด
min_score_ds,None,คะแนนต่ำสุด หลังประมวลผลรอบ 2,คะแนนต่ำสุด ประมวลผลครั้งที่ 2,คะแนนต่ำสุด DS


## 2. Load

One tidy frame, one schema. Blank `รายละเอียด` / `สาขา/วิชาเอก` become `""`
rather than `NaN` — pandas' `groupby` drops NaN keys by default, so leaving
them would silently discard ~50 Chula rows a year from any grouped
aggregation (2,838 of TCAS69's 7,639 seats).

In [3]:
df = load_all()
print(f"{len(df):,} rows x {len(df.columns)} cols")
df.groupby("year").size().rename("rows").to_frame().T

19,437 rows x 17 cols


year,66,67,68,69
rows,4670,4718,4945,5104


In [4]:
df.head(3)[["year", "univ", "code", "faculty", "program",
            "applied", "passed", "min_score", "max_score"]]

,year,univ,code,faculty,program,applied,passed,min_score,max_score
0,66,จุฬาลงกรณ์มหาวิทยาลัย,10010121300001A,คณะวิศวกรรมศาสตร์,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมศา...,2424.0,360.0,54.1500,80.8274
1,66,จุฬาลงกรณ์มหาวิทยาลัย,10010121300501A,คณะวิศวกรรมศาสตร์,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมคอ...,1422.0,65.0,71.4166,86.5610
2,66,จุฬาลงกรณ์มหาวิทยาลัย,10010121300599A,คณะวิศวกรรมศาสตร์,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมคอ...,2305.0,185.0,61.1666,84.9052


## 3. Assert

`validate()` raises on any breach of the invariants everything downstream
rests on, and *returns* the things that are logged instead:

| asserted (raises) | logged (returned) |
|---|---|
| row counts per year | national row-key collisions |
| row key unique **within Chula** | national `min_score == 0` with real admits |
| Chula `min_score == 0` ⟹ `passed == 0` | national `min > max` |
| Chula `min_score ≤ max_score` | TCAS66's 46 no-admit/nonzero-floor rows |
| Chula's 14 faculty names identical across years | |

The split is deliberate: the national rows are legitimate data or genuine ทปอ.
defects, and deleting them to make an assert pass would be the wrong fix.

In [5]:
report = validate(df)
pd.Series({k: v for k, v in report.items() if k != "chula_faculties"}).to_frame("n")

,n
national_min_gt_max_66,0
national_key_collisions_66,27
national_zero_min_with_admits_66,200
national_no_admits_nonzero_min_66,46
national_min_gt_max_67,1
national_key_collisions_67,34
national_zero_min_with_admits_67,200
national_no_admits_nonzero_min_67,0
national_min_gt_max_68,0
national_key_collisions_68,32


Two source defects worth naming, both left uncorrected and visible:

- **TCAS67, ราชวิทยาลัยจุฬาภรณ์ `10320104112101A`** publishes `min = 46.33`,
  `max = 10.0`. Its own DS columns read `min = 40.86`, `max = 46.33`, so the
  real max is 46.33 and the `10.0` is a typo. One row, not Chula.
- **TCAS66 has 46 national rows** reporting no admits beside a nonzero floor.
  TCAS67–69 have none. Rule adopted: leave them in place and let the standard
  `passed > 0` validity filter exclude them, like any other unfilled program.

## 4. The three traps, in one cell each

Demonstrated numerically here so the loader's design choices are legible;
`nb/02_explore` takes them apart properly. All three are locked in
`tests/test_traps.py`.

### Trap 1 — `รับ` (seats) cannot be summed

In [6]:
c69 = chula(df[df["year"] == 69])
pd.Series({
    "sum() raw":                 c69["seats"].sum(),
    "groupby(code).first()":     c69.groupby("code")["seats"].first().sum(),
    "reconstructed_seats()":     reconstructed_seats(c69),
    "admits (ผ่าน)":              c69["passed"].sum(),
}).to_frame("TCAS69 Chula seats")

,TCAS69 Chula seats
sum() raw,7639.0
groupby(code).first(),2728.0
reconstructed_seats(),3847.0
admits (ผ่าน),3989.0


Every rule fails. `sum()` double-counts quotas shared across
(สาขา × เลือกสอบวิชา) rows; `first()` throws away a real pool — code
`10010122904301A` (อักษรศาสตร์) holds *two* pools, 20 and 209, and `first()`
keeps 20. The best available reconstruction still lands **below** the number
of students actually admitted, in all four years.

**The published file does not permit an exact seat count.** Use `ผ่าน`
(admits) as the denominator; report competition as *applications per admit*.
`reconstructed_seats()` is the only sanctioned aggregation of `seats`, and it
carries that warning in its docstring.

In [7]:
seats = pd.DataFrame({
    "reconstructed": {y: reconstructed_seats(chula(df[df.year == y])) for y in (66, 67, 68, 69)},
    "admits":        chula(df).groupby("year")["passed"].sum(),
})
seats["shortfall"] = seats["reconstructed"] - seats["admits"]
seats

,reconstructed,admits,shortfall
66,4036.0,4118.0,-82.0
67,3837.0,3916.0,-79.0
68,3815.0,4007.0,-192.0
69,3847.0,3989.0,-142.0


### Trap 2 — `min_score == 0` means "nobody admitted" at Chula, but not nationally

In [8]:
ch = chula(df)
zero_rule = pd.DataFrame({
    "chula min==0":            ch[ch.min_score == 0].groupby("year").size(),
    "...of which passed==0":   ch[(ch.min_score == 0) & (ch.passed == 0)].groupby("year").size(),
    "national min==0 & passed>0": df[(df.min_score == 0) & (df.passed > 0)].groupby("year").size(),
})
zero_rule

,chula min==0,...of which passed==0,national min==0 & passed>0
year,,,
66,12,12,200
67,10,10,200
68,16,16,210
69,19,19,161


At Chula the rule is clean — 12/10/16/19 rows, 100% of them with zero admits.
Nationally it is false: 161–210 rows a year have a genuine zero floor beside
real admits, mostly private and Rajabhat universities whose Admission criteria
are not exam-scored.

**So filter on `passed > 0` for validity — never on `min_score > 0`.**
The national median moves about a point between the two filters; the *shape*
is identical, so conclusions about direction are safe and conclusions about
level are not. Both must be reported.

In [9]:
pd.DataFrame({
    "passed>0":         df[df.passed > 0].groupby("year")["min_score"].median(),
    "passed>0 & min>0": df[(df.passed > 0) & (df.min_score > 0)].groupby("year")["min_score"].median(),
}).round(2)

,passed>0,passed>0 & min>0
year,,
66,50.64,51.71
67,51.75,52.64
68,53.85,54.70
69,52.66,53.33


At **Chula** the two filters select exactly the same rows — a direct
consequence of the rule above. That is why every Chula figure in this project
is "stable under both filters": there is only one filter.

In [10]:
print(len(ch[ch.passed > 0]), "==", len(ch[(ch.passed > 0) & (ch.min_score > 0)]))

483 == 483


### Trap 3 — most universities score in raw percent, a few in T-Score

The composite is `Σ (raw_subject ÷ 100) × weight` with weights summing to 100,
so published min/max sit on a 0–100 raw-weighted scale. But KMITL, Chiang Mai
and Thammasat (plus Silpakorn and RMUTT in some years) specify **T-Score** in
their เกณฑ์, so their numbers are T-Score-weighted composites.

Chula is raw. Any cross-university comparison touching that set is
apples-to-oranges and must be marked in-chart, not silently plotted — handled
in `nb/04`. The names are pinned in `src.load.TSCORE_UNIVERSITIES`.

## 5. Panel

Faculty names are **asserted** identical across years, not crosswalked — the
assert above already covers it. All 48 TCAS66 Chula codes persist to TCAS69,
where 2 new ones appear; the balanced panel keeps the 48.

In [11]:
codes = {y: set(g["code"]) for y, g in ch.groupby("year")}
print(f"stable codes present in all four years: {len(stable_chula_codes(df))}")
print(f"all TCAS66 codes survive to TCAS69:     {codes[66] <= codes[69]}")
print(f"new codes appearing in TCAS69:          {len(codes[69] - codes[66])}")
print(f"Chula faculties (identical 66->69):     {len(report['chula_faculties'])}")

stable codes present in all four years: 48
all TCAS66 codes survive to TCAS69:     True
new codes appearing in TCAS69:          2
Chula faculties (identical 66->69):     14


In [12]:
panel = chula_panel(df)
print(f"panel: {panel.shape[0]} rows = {panel.code.nunique()} codes x {panel.year.nunique()} years")
panel.head(4)

panel: 192 rows = 48 codes x 4 years


,code,year,faculty,program,n_rows,applied,passed,min_score,min_score_median,max_score,n_scored_rows,apps_per_admit
0,10010127703701A,66,คณะครุศาสตร์,หลักสูตรครุศาสตรบัณฑิต,28,8094.0,369.0,45.5665,68.91630,92.2220,23,21.934959
1,10010127703701A,67,คณะครุศาสตร์,หลักสูตรครุศาสตรบัณฑิต,28,8073.0,322.0,48.4944,68.71935,88.1110,22,25.071429
2,10010127703701A,68,คณะครุศาสตร์,หลักสูตรครุศาสตรบัณฑิต,28,4231.0,318.0,51.5054,70.59700,90.4028,21,13.305031
3,10010127703701A,69,คณะครุศาสตร์,หลักสูตรครุศาสตรบัณฑิต,27,4662.0,290.0,43.4374,73.63530,92.3118,21,16.075862


A code fans out into several (สาขา × เลือกสอบวิชา) rows, so the panel
aggregates: `applied`/`passed` are summed, and score aggregates use **only
rows with `passed > 0`** so a no-admit row's zero floor cannot drag the
program's floor down. `seats` is deliberately absent. Panel `applied` for 69
is 48,235 rather than 48,615 because the two new-in-69 codes are excluded.

## 6. Write

In [13]:
pq, csv = write(df)
panel_path = PROCESSED / "chula_panel.csv"
panel.to_csv(panel_path, index=False)
for p in (pq, csv, panel_path):
    print(f"{p.name:28s} {p.stat().st_size / 1024:8.1f} KB")

admission_long.parquet          824.5 KB
admission_long.csv             8063.6 KB
chula_panel.csv                  48.4 KB


## What this notebook establishes

- The four files sit on **one schema** with no fuzzy matching: 14 Chula
  faculty names byte-identical across four years, 48 program codes stable.
- **Seats are unusable.** Every dedup rule fails and the best reconstruction
  falls below actual admits in all four years. Competition is measured as
  applications per admit.
- **`min_score == 0` is a Chula-only "nobody admitted" signal.** Nationally
  161–210 rows a year contradict it. Validity filters on `passed > 0`.
- Two source defects are logged, not patched: one TCAS67 inverted min/max, and
  46 TCAS66 rows with no admits beside a nonzero floor.

Next: `nb/02_explore` takes the traps apart with the national picture, then
`nb/03_normalize` handles exam-scale drift.